In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import log_loss,accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder


In [14]:
# involving columns transformer

In [15]:
hr=pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")
X , y = hr.drop('left',axis=1) , hr['left']


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().set_output(transform="pandas")

In [17]:
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")

In [24]:
from sklearn.linear_model import LogisticRegression


lr=LogisticRegression()

In [25]:



from sklearn.pipeline import Pipeline


scores=[]
for percent_feature in np.linspace(.5,.95,10):#percentage colums from 50% from 
    prcomp=PCA(n_components=percent_feature).set_output(transform='pandas')
    pipe=Pipeline([("Trans_col",col_transformer),("SCL",scaler),('PCA',prcomp),("Lr",lr)])
    pipe.fit(X_train,y_train)
    y_pred_proba = pipe.predict_proba(X_test)
    scores.append([percent_feature,log_loss(y_test, y_pred_proba)*100,accuracy_score(y_test, y_pred_proba)*100])
per_feat_df=pd.DataFrame(scores,columns=('percentage_feature','log_loss_score','accuracy_score'))
per_feat_df=per_feat_df.sort_values('log_loss_score')
per_feat_df
pipe

ValueError: Classification metrics can't handle a mix of binary and continuous-multioutput targets

In [26]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Load data
hr = pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")
X, y = hr.drop('left', axis=1), hr['left']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Setup transformers
scaler = StandardScaler().set_output(transform="pandas")
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")

col_transformer = ColumnTransformer(
    [("OHE", ohe, make_column_selector(dtype_include=object))], 
    remainder='passthrough', 
    verbose_feature_names_out=False
).set_output(transform="pandas")

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)  # Increased max_iter to ensure convergence

# Test different PCA variance thresholds
scores = []
for percent_feature in np.linspace(0.5, 0.95, 10):
    prcomp = PCA(n_components=percent_feature).set_output(transform='pandas')
    pipe = Pipeline([
        ("Trans_col", col_transformer),
        ("SCL", scaler),
        ('PCA', prcomp),
        ("Lr", lr)
    ])
    
    # Fit pipeline
    pipe.fit(X_train, y_train)
    
    # Get predictions
    y_pred_proba = pipe.predict_proba(X_test)
    y_pred = pipe.predict(X_test)  # Get actual class predictions
    
    # Calculate metrics
    ll_score = log_loss(y_test, y_pred_proba)
    acc_score = accuracy_score(y_test, y_pred)
    
    # Get actual number of components used
    n_components_used = pipe.named_steps['PCA'].n_components_
    
    scores.append([
        percent_feature, 
        n_components_used,
        ll_score, 
        acc_score
    ])

# Create results dataframe
per_feat_df = pd.DataFrame(
    scores, 
    columns=['variance_explained', 'n_components', 'log_loss', 'accuracy']
)
per_feat_df = per_feat_df.sort_values('log_loss')

print("PCA Performance Results:")
print("=" * 70)
print(per_feat_df.to_string(index=False))
print("\n" + "=" * 70)
print(f"\nBest configuration (lowest log loss):")
print(f"  Variance explained: {per_feat_df.iloc[0]['variance_explained']:.2f}")
print(f"  Components used: {per_feat_df.iloc[0]['n_components']:.0f}")
print(f"  Log loss: {per_feat_df.iloc[0]['log_loss']:.4f}")
print(f"  Accuracy: {per_feat_df.iloc[0]['accuracy']:.4f}")

# Final pipeline with best configuration
best_variance = per_feat_df.iloc[0]['variance_explained']
final_pipe = Pipeline([
    ("Trans_col", col_transformer),
    ("SCL", scaler),
    ('PCA', PCA(n_components=best_variance).set_output(transform='pandas')),
    ("Lr", LogisticRegression(max_iter=1000))
])
final_pipe.fit(X_train, y_train)

PCA Performance Results:
 variance_explained  n_components  log_loss  accuracy
               0.90            14  0.453497  0.772591
               0.95            15  0.453767  0.770924
               0.80            12  0.455222  0.776592
               0.85            13  0.455422  0.776926
               0.75            11  0.456850  0.775925
               0.70            10  0.461069  0.780927
               0.60             9  0.461081  0.780927
               0.65             9  0.461081  0.780927
               0.55             8  0.463336  0.778926
               0.50             7  0.463372  0.778926


Best configuration (lowest log loss):
  Variance explained: 0.90
  Components used: 14
  Log loss: 0.4535
  Accuracy: 0.7726


Pipeline(steps=[('Trans_col',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OHE',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000013BCFB5AE90>)],
                                   verbose_feature_names_out=False)),
                ('SCL', StandardScaler()),
                ('PCA', PCA(n_components=np.float64(0.8999999999999999))),
                ('Lr', LogisticRegression(max_iter=1000))])